# Chapter 20 – Worked examples

This notebook runs the examples of [Chapter 20](https://katra.ballardini.com.ar/power-grid-topology-optimization-course/devdev/part-3-power-systems/ch20-protection-short-circuit/) live. The code lives in the modules next to it
(`fault_levels.py`, `station_sc.py`, `rx_plane.py`); the cells only call them, so a change in a module is a
change here too.

Run it from this folder (the kernel's working directory must be `src/`), or open it in Jupyter Lab from the
repository.

In [ ]:
# JupyterLite only: install the libraries this notebook and its chapter modules use.
%pip install -q colorama==0.4.6 deepdiff==8.6.2 geojson==3.3.0 networkx numpy packaging pandas pandera==0.34.1 scipy tqdm typing_extensions tzdata
%pip install -q --no-deps pandapower==3.5.6

In [ ]:
import fault_levels
import numpy as np
import rx_plane
import station_sc

## Fault levels of a double-busbar station by hand

Two 220 kV infeeds ($S''_{k} = 8\,000$ and $6\,000$ MVA) over 40 km and 60 km of $j0.4\ \Omega/\text{km}$,
$c = 1.1$. With the coupler closed the two paths are in parallel; open, each busbar sees its own infeed.

In [ ]:
for key, value in fault_levels.worked_example().items():
    print(f'{key:22s} {value:8.3f}')

## The same station in pandapower

`station_sc.station(coupler_closed, extra_paths, r_ohm_per_km)` builds the grid; without the extra paths and
with lossless lines `calc_sc` reproduces the hand calculation. The extra paths (two lines to a load bus and a
150 km line between the infeeds) raise the split fault levels.

In [ ]:
print(station_sc.station_table(extra_paths=False, r_ohm_per_km=0.0).round(3))
print(station_sc.station_table().round(3))

## ToOp's distance-protection polygon

The relay table of `docs/contingency_analysis/cascade.md` gives `angle` 80°, `r_i` 2.5 Ω, `r_v` 10 Ω,
`x_v` 15 Ω; `_build_poly` then self-intersects because $r_v \tan 80° = 56.7\ \Omega > x_v$. With 30°, the
value the unit tests use, the polygon is simple.

In [ ]:
for angle in (30.0, 80.0):
    poly = rx_plane.toop_polygon(angle, 2.5, 10.0, 15.0)
    print(angle, 'deg: valid', rx_plane.is_valid(angle, 2.5, 10.0, 15.0), 'simple', rx_plane.is_simple(poly))
    print('   vertices', [(round(float(r), 2), round(float(x), 2)) for r, x in poly])

The apparent impedance of a 380 kV line at $\cos\varphi = 0.95$ enters the 30° polygon only above about
55 GVA, so these example parameters are not meant for a 380 kV line in primary ohms.

In [ ]:
phi = np.degrees(np.arccos(0.95))
poly = rx_plane.toop_polygon(30.0, 2.5, 10.0, 15.0)
print(round(rx_plane.tripping_mva(poly, 380.0, phi)), 'MVA')
print(round(rx_plane.tripping_mva(poly, 380.0, phi, 1.41)), 'MVA with the 1.41 factor')